# Shown Space Scoring Path Visuals

This notebook uses Shown Space's public game API to recreate field-path visuals from coordinates, then summarizes a team's scoring possessions as an interactive average path and heatmap.

Default team: `glory`.

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd

from ufa import (
    average_scoring_path,
    build_scoring_possessions,
    cluster_scoring_possessions,
    create_scoring_possession_browser,
    create_team_scoring_possession_browser,
    create_team_playstyle_report_browser,
    fetch_shownspace_games,
    fetch_shownspace_season_throws,
    fetch_shownspace_throws_for_games,
    plot_average_scoring_path,
    plot_possession_path,
    plot_representative_paths,
    plot_scoring_heatmap,
    plot_team_representative_path_grid,
    select_representative_paths,
    select_top_paths,
    summarize_path_clusters,
    summarize_team_playstyle,
    summarize_team_playstyles,
)

## Settings

Use `MAX_GAMES = 3` first to validate the visual quickly. Set `MAX_GAMES = None` for the full team season.

In [2]:
SEASON = 2026
TEAM_ID = "spiders"
MAX_GAMES = 3
SAMPLE_GAMES_RANDOMLY = True
RANDOM_STATE = 7
UNIQUE_REPRESENTATIVE_GAMES = True
PULL_RECEIVE_SCORES_ONLY = True
LONG_FIELD_ONLY = True
MAX_START_Y = 45
MIN_FIELD_PROGRESS = 50
EXCLUDE_HUCKS_FROM_TOP_PATHS = True

all_games = fetch_shownspace_games(season=SEASON, final_only=True)
team_games = all_games[
    all_games["HomeTeamID"].str.lower().eq(TEAM_ID.lower())
    | all_games["AwayTeamID"].str.lower().eq(TEAM_ID.lower())
].reset_index(drop=True)

if MAX_GAMES is None:
    games = team_games.copy()
elif SAMPLE_GAMES_RANDOMLY:
    games = (
        team_games
        .sample(n=min(MAX_GAMES, len(team_games)), random_state=RANDOM_STATE)
        .sort_values("StartTimestamp")
        .reset_index(drop=True)
    )
else:
    games = team_games.head(MAX_GAMES).copy()

throws = fetch_shownspace_throws_for_games(games["GameID"].tolist(), delay=0.15)

games[["GameID", "AwayTeamID", "HomeTeamID", "AwayScore", "HomeScore", "Status", "StartTimestamp"]]


,GameID,AwayTeamID,HomeTeamID,AwayScore,HomeScore,Status,StartTimestamp
0,2026-04-26-ORE-OAK,steel,spiders,11,34,Final,2026-04-26 15:30:00
1,2026-05-30-SD-OAK,growlers,spiders,17,23,Final,2026-05-30 15:00:00
2,2026-06-26-OAK-COL,spiders,apex,26,17,Final,2026-06-26 19:00:00


In [3]:
possessions, paths = build_scoring_possessions(throws, team_id=TEAM_ID)

print(f"Throws loaded: {len(throws):,}")
print(f"Scoring possessions found for {TEAM_ID}: {len(possessions):,}")

possessions.sort_values("risk_adjusted_aec_per_throw", ascending=False).head(10)

Throws loaded: 1,616
Scoring possessions found for spiders: 79


,possession_id,GameID,team_id,start_timestamp,game_quarter,quarter_point,possession_num,is_home_team,line_type,start_x,...,mean_cp,risk_adjusted_aec_per_throw,total_yards,yards_per_throw,total_throw_distance,avg_throw_distance,max_throw_distance,huck_count,reset_count,lateral_yards
22,2026-04-26-ORE-OAK|3|8|2|True,2026-04-26-ORE-OAK,spiders,2026-04-26 15:30:00,3,8,2,True,d_line,-8.37,...,0.976773,0.976773,3.96,3.960000,4.609902,4.609902,4.609902,0,0,2.36
11,2026-04-26-ORE-OAK|2|3|2|True,2026-04-26-ORE-OAK,spiders,2026-04-26 15:30:00,2,3,2,True,d_line,6.19,...,0.972435,0.972435,6.91,6.910000,6.916083,6.916083,6.916083,0,0,0.29
59,2026-06-26-OAK-COL|1|1|1|False,2026-06-26-OAK-COL,spiders,2026-06-26 19:00:00,1,1,1,False,o_line,-2.41,...,0.939296,0.939296,12.36,12.360000,16.006202,16.006202,16.006202,0,0,10.17
33,2026-04-26-ORE-OAK|4|11|2|True,2026-04-26-ORE-OAK,spiders,2026-04-26 15:30:00,4,11,2,True,d_line,-4.04,...,0.904849,0.455642,40.23,20.115000,45.505472,22.752736,32.754641,0,0,20.67
44,2026-05-30-SD-OAK|2|6|2|True,2026-05-30-SD-OAK,spiders,2026-05-30 15:00:00,2,6,2,True,o_line,-26.41,...,0.723512,0.361781,73.87,36.935000,88.650771,44.325385,48.575810,2,0,37.27
6,2026-04-26-ORE-OAK|1|10|7|True,2026-04-26-ORE-OAK,spiders,2026-04-26 15:30:00,1,10,7,True,o_line,11.02,...,0.968338,0.327513,14.16,4.720000,32.033977,10.677992,14.367387,0,0,22.04
37,2026-05-30-SD-OAK|1|5|1|True,2026-05-30-SD-OAK,spiders,2026-05-30 15:00:00,1,5,1,True,o_line,0.00,...,0.827096,0.275767,77.61,25.870000,104.955642,34.985214,64.900408,1,1,21.00
24,2026-04-26-ORE-OAK|3|11|5|True,2026-04-26-ORE-OAK,spiders,2026-04-26 15:30:00,3,11,5,True,o_line,0.00,...,0.942110,0.259848,65.54,16.385000,82.033228,20.508307,24.020075,0,0,36.31
72,2026-06-26-OAK-COL|4|1|2|False,2026-06-26-OAK-COL,spiders,2026-06-26 19:00:00,4,1,2,False,d_line,-7.23,...,0.962589,0.255588,22.73,5.682500,50.338383,12.584596,21.485237,0,1,28.98
15,2026-04-26-ORE-OAK|2|9|2|True,2026-04-26-ORE-OAK,spiders,2026-04-26 15:30:00,2,9,2,True,o_line,14.16,...,0.957518,0.255139,26.66,8.886667,43.988041,14.662680,19.306706,0,1,8.37


## Long-Field Possession Filter

Use this to focus the visuals on possessions that start farther from the scoring end zone, instead of short-field scores after turnovers.

In [4]:
analysis_possessions = possessions.copy()

if PULL_RECEIVE_SCORES_ONLY:
    analysis_possessions = analysis_possessions[
        analysis_possessions["possession_num"].eq(1)
    ].copy()

if LONG_FIELD_ONLY:
    analysis_possessions = analysis_possessions[
        analysis_possessions["start_y"].le(MAX_START_Y)
        & analysis_possessions["field_progress"].ge(MIN_FIELD_PROGRESS)
    ].copy()

analysis_ids = set(analysis_possessions["possession_id"])
analysis_paths = [
    path for path in paths
    if path["possession_id"].iloc[0] in analysis_ids
]

print(f"All scoring possessions: {len(possessions):,}")
initial_scoring_holds = possessions["possession_num"].eq(1).sum()
print(f"Initial-possession scoring holds: {initial_scoring_holds:,}")
print(f"Analysis possessions: {len(analysis_possessions):,}")

analysis_possessions[[
    "possession_id", "GameID", "possession_num", "start_y", "end_y",
    "field_progress", "throw_count", "total_aec", "aec_per_throw"
]].head(10)


All scoring possessions: 79
Initial-possession scoring holds: 34
Analysis possessions: 27


,possession_id,GameID,possession_num,start_y,end_y,field_progress,throw_count,total_aec,aec_per_throw
0,2026-04-26-ORE-OAK|1|2|1|True,2026-04-26-ORE-OAK,1,20.00,107.60,87.60,9,1.239444,0.137716
3,2026-04-26-ORE-OAK|1|6|1|True,2026-04-26-ORE-OAK,1,17.90,114.98,97.08,8,1.002001,0.125250
9,2026-04-26-ORE-OAK|2|1|1|True,2026-04-26-ORE-OAK,1,20.00,108.49,88.49,12,1.107608,0.092301
18,2026-04-26-ORE-OAK|3|3|1|True,2026-04-26-ORE-OAK,1,31.08,105.83,74.75,5,0.995502,0.199100
21,2026-04-26-ORE-OAK|3|7|1|True,2026-04-26-ORE-OAK,1,44.16,104.55,60.39,11,1.001043,0.091004
25,2026-04-26-ORE-OAK|4|1|1|True,2026-04-26-ORE-OAK,1,19.77,111.24,91.47,9,0.993099,0.110344
28,2026-04-26-ORE-OAK|4|5|1|True,2026-04-26-ORE-OAK,1,4.52,103.96,99.44,7,0.991230,0.141604
34,2026-04-26-ORE-OAK|4|13|1|True,2026-04-26-ORE-OAK,1,40.91,104.45,63.54,15,1.352845,0.090190
36,2026-05-30-SD-OAK|1|3|1|True,2026-05-30-SD-OAK,1,40.00,103.80,63.80,9,0.965933,0.107326
37,2026-05-30-SD-OAK|1|5|1|True,2026-05-30-SD-OAK,1,30.00,107.61,77.61,3,1.000248,0.333416


## Glory Scoring Possession Browser

This is the Shown Space-style possession browser. It uses a custom HTML/SVG field instead of Plotly so the field has simple lines, dots, hover tooltips, and no chart toolbar.

In [5]:
BROWSER_SEASON = 2026
BROWSER_TEAM_ID = "glory"
BROWSER_MAX_GAMES = None  # None means every final game for the team

# Defaults to all Glory scoring possessions. Turn these on only if you want a narrower view.
BROWSER_PULL_RECEIVE_SCORES_ONLY = False
BROWSER_LONG_FIELD_ONLY = False
BROWSER_MAX_START_Y = 45
BROWSER_MIN_FIELD_PROGRESS = 50
BROWSER_EXCLUDE_HUCKS = False

browser_all_games = fetch_shownspace_games(season=BROWSER_SEASON, final_only=True)
browser_games = browser_all_games[
    browser_all_games["HomeTeamID"].str.lower().eq(BROWSER_TEAM_ID.lower())
    | browser_all_games["AwayTeamID"].str.lower().eq(BROWSER_TEAM_ID.lower())
].sort_values("StartTimestamp").reset_index(drop=True)

if BROWSER_MAX_GAMES is not None:
    browser_games = browser_games.head(BROWSER_MAX_GAMES).copy()

browser_throws = fetch_shownspace_throws_for_games(
    browser_games["GameID"].tolist(),
    delay=0.15,
)
browser_possessions, browser_paths = build_scoring_possessions(
    browser_throws,
    team_id=BROWSER_TEAM_ID,
)

if BROWSER_PULL_RECEIVE_SCORES_ONLY:
    browser_possessions = browser_possessions[
        browser_possessions["possession_num"].eq(1)
    ].copy()

if BROWSER_LONG_FIELD_ONLY:
    browser_possessions = browser_possessions[
        browser_possessions["start_y"].le(BROWSER_MAX_START_Y)
        & browser_possessions["field_progress"].ge(BROWSER_MIN_FIELD_PROGRESS)
    ].copy()

if BROWSER_EXCLUDE_HUCKS:
    browser_possessions = browser_possessions[
        browser_possessions["huck_count"].fillna(0).eq(0)
    ].copy()

browser_ids = set(browser_possessions["possession_id"])
browser_paths = [
    path for path in browser_paths
    if path["possession_id"].iloc[0] in browser_ids
]

print(f"Games loaded: {len(browser_games):,}")
print(f"Throws loaded: {len(browser_throws):,}")
print(f"Browser possessions: {len(browser_possessions):,}")

browser_possessions[[
    "possession_id", "GameID", "game_quarter", "quarter_point",
    "possession_num", "throw_count", "start_y", "end_y",
    "field_progress", "total_aec", "aec_per_throw"
]].head(10)


Games loaded: 10
Throws loaded: 5,550
Browser possessions: 240


,possession_id,GameID,game_quarter,quarter_point,possession_num,throw_count,start_y,end_y,field_progress,total_aec,aec_per_throw
0,2026-04-25-DC-BOS|1|1|2|True,2026-04-25-DC-BOS,1,1,2,2,75.03,102.12,27.09,1.010065,0.505033
1,2026-04-25-DC-BOS|1|3|1|True,2026-04-25-DC-BOS,1,3,1,10,14.77,106.96,92.19,0.987499,0.098750
2,2026-04-25-DC-BOS|1|4|2|True,2026-04-25-DC-BOS,1,4,2,10,34.12,115.29,81.17,1.145452,0.114545
3,2026-04-25-DC-BOS|1|5|2|True,2026-04-25-DC-BOS,1,5,2,20,19.74,104.06,84.32,0.891622,0.044581
4,2026-04-25-DC-BOS|1|6|2|True,2026-04-25-DC-BOS,1,6,2,7,49.80,108.32,58.52,1.004663,0.143523
5,2026-04-25-DC-BOS|1|8|1|True,2026-04-25-DC-BOS,1,8,1,9,46.70,115.16,68.46,1.000726,0.111192
6,2026-04-25-DC-BOS|1|9|2|True,2026-04-25-DC-BOS,1,9,2,4,44.51,116.90,72.39,-0.193256,-0.048314
7,2026-04-25-DC-BOS|2|1|1|True,2026-04-25-DC-BOS,2,1,1,15,8.45,105.29,96.84,1.041824,0.069455
8,2026-04-25-DC-BOS|2|2|2|True,2026-04-25-DC-BOS,2,2,2,3,89.80,105.61,15.81,1.669016,0.556339
9,2026-04-25-DC-BOS|2|4|1|True,2026-04-25-DC-BOS,2,4,1,10,13.80,114.32,100.52,0.984443,0.098444


## Widget Display Check

Run this small check if the browser output looks blank. If the slider does not appear, restart the kernel and rerun the import cell.

In [6]:
import ipywidgets as widgets
widgets.IntSlider(description="widget test")


IntSlider(value=0, description='widget test')

In [7]:
import sys
sys.path.insert(0, "../src")

import importlib
import ufa.shownspace_paths as shownspace_paths
importlib.reload(shownspace_paths)

create_scoring_possession_browser = shownspace_paths.create_scoring_possession_browser
create_team_scoring_possession_browser = shownspace_paths.create_team_scoring_possession_browser

In [8]:
from IPython.display import display

team_browser = create_team_scoring_possession_browser(
    season=BROWSER_SEASON,
    default_team_id=BROWSER_TEAM_ID,
    final_only=True,
    max_games=BROWSER_MAX_GAMES,
    pull_receive_scores_only=BROWSER_PULL_RECEIVE_SCORES_ONLY,
    long_field_only=BROWSER_LONG_FIELD_ONLY,
    max_start_y=BROWSER_MAX_START_Y,
    min_field_progress=BROWSER_MIN_FIELD_PROGRESS,
    exclude_hucks=BROWSER_EXCLUDE_HUCKS,
    n_shape_clusters=8,
)

display(team_browser)


## Team Playstyle Summary Report

These summaries use the same scoring-possession shape features as the browser. The text is rule-based, so every phrase traces back to the metrics shown in the table.


In [9]:
PLAYSTYLE_COLUMNS = [
    "team_id",
    "possessions",
    "primary_shapes",
    "attack_spaces",
    "pace_style",
    "field_width_style",
    "huck_usage",
    "reset_usage",
    "efficiency_note",
    "playstyle_summary",
]

BACKING_METRIC_COLUMNS = [
    "avg_throws",
    "avg_width",
    "avg_side_switches",
    "avg_directness",
    "avg_middle_usage",
    "avg_sideline_usage",
    "avg_hucks",
    "avg_resets",
    "avg_aec_per_throw",
    "avg_cp",
]

team_playstyle = summarize_team_playstyle(
    browser_possessions,
    browser_paths,
    team_id=BROWSER_TEAM_ID,
    n_shape_clusters=8,
)

team_playstyle_table = pd.DataFrame([team_playstyle])

display(
    create_team_playstyle_report_browser(
        team_playstyle,
        team_playstyle_table,
        title=f"{BROWSER_TEAM_ID.title()} playstyle summary, {BROWSER_SEASON}",
    )
)


HTML(value='\n    <div class="ufa-playstyle-browser">\n      <style>\n        .ufa-playstyle-browser {\n      …

In [7]:
PLAYSTYLE_TEAM_IDS = ["glory", "empire", "spiders"]
PLAYSTYLE_MAX_GAMES = None

playstyle_rows = []
for playstyle_team_id in PLAYSTYLE_TEAM_IDS:
    team_games = browser_all_games[
        browser_all_games["HomeTeamID"].str.lower().eq(playstyle_team_id.lower())
        | browser_all_games["AwayTeamID"].str.lower().eq(playstyle_team_id.lower())
    ].sort_values("StartTimestamp").reset_index(drop=True)

    if PLAYSTYLE_MAX_GAMES is not None:
        team_games = team_games.head(PLAYSTYLE_MAX_GAMES).copy()

    team_throws = fetch_shownspace_throws_for_games(
        team_games["GameID"].tolist(),
        delay=0.15,
    )
    team_possessions, team_paths = build_scoring_possessions(
        team_throws,
        team_id=playstyle_team_id,
    )
    playstyle_rows.append(
        summarize_team_playstyle(
            team_possessions,
            team_paths,
            team_id=playstyle_team_id,
            n_shape_clusters=8,
        )
    )

team_playstyle_table = pd.DataFrame(playstyle_rows)
selected_team_row = team_playstyle_table[
    team_playstyle_table["team_id"].astype(str).str.lower().eq(BROWSER_TEAM_ID.lower())
]
if selected_team_row.empty:
    selected_team_playstyle = team_playstyle_table.iloc[0]
else:
    selected_team_playstyle = selected_team_row.iloc[0]

display(
    create_team_playstyle_report_browser(
        selected_team_playstyle,
        team_playstyle_table,
        title=f"Team playstyle comparison, {BROWSER_SEASON}",
    )
)


HTML(value='\n    <div class="ufa-playstyle-browser">\n      <style>\n        .ufa-playstyle-browser {\n      …

## Average Scoring Path

The average path is progress-normalized. Each scoring possession is resampled to fixed progress checkpoints from possession start to goal, then the checkpoint coordinates are averaged.

In [ ]:
avg_path = average_scoring_path(paths)
avg_path

,checkpoint,x,y,mean_cumulative_aec,mean_cp,mean_win_prob,possessions
0,0.0,0.152911,41.605949,0.014659,0.949453,0.699823,79
1,0.2,0.902229,54.783216,0.166711,0.951991,0.700406,79
2,0.4,1.930303,66.845432,0.321297,0.944712,0.701383,79
3,0.6,-0.271007,79.692394,0.505429,0.920802,0.702353,79
4,0.8,-1.003799,92.662060,0.709294,0.899952,0.703530,79
5,1.0,-1.103418,105.923165,0.983075,0.897264,0.705000,79


In [ ]:
fig = plot_average_scoring_path(
    avg_path,
    paths=paths,
    title=f"{TEAM_ID.title()} average scoring path, {SEASON} sample",
    show_individual_paths=True,
)
fig.show()

## Real Representative Paths

The mean path is useful as a center-of-gravity check, but it can hide the actual bends, resets, hucks, and lateral movement that make an offense interesting. These cells keep real possessions intact and then pick examples worth studying.

In [ ]:
clustered_possessions = cluster_scoring_possessions(analysis_possessions, analysis_paths, n_clusters=4)
cluster_summary = summarize_path_clusters(clustered_possessions)
cluster_summary

,path_cluster,style,possessions,avg_throws,avg_aec_per_throw,avg_cp,avg_yards_per_throw,avg_max_throw_distance,avg_resets,avg_lateral_yards,avg_width,avg_directness,avg_side_switches,avg_middle_third_share,avg_sideline_share,avg_red_zone_entry_x
0,0,methodical,6,8.833333,0.115057,0.962863,9.078778,22.378839,0.833333,74.391667,39.311667,0.649456,2.000000,0.312269,0.314120,-5.203333
6,2,reset-heavy,5,16.400000,0.070956,0.967650,5.028749,26.864359,4.800000,150.064000,35.762000,0.371665,5.000000,0.572059,0.090098,9.032000
7,3,huck score,3,3.666667,0.288325,0.757280,20.787778,59.605254,0.666667,29.583333,22.040000,0.831836,0.666667,0.666667,0.088889,3.970000
3,1,methodical,3,10.333333,0.101912,0.950920,8.632500,23.678032,0.666667,90.823333,35.806667,0.654797,1.666667,0.372222,0.239815,-4.920000
5,2,methodical,3,11.000000,0.080359,0.962313,7.647778,22.550765,2.000000,102.973333,39.756667,0.554391,2.666667,0.365741,0.370370,22.290000
4,1,mixed,2,6.500000,0.154726,0.962925,15.117857,20.530608,0.000000,27.305000,17.795000,0.933794,2.000000,0.601190,0.000000,8.585000
8,3,methodical,2,8.000000,0.130075,0.944798,10.323125,34.630134,2.000000,73.295000,37.170000,0.568843,1.500000,0.562500,0.093750,-17.190000
2,1,huck score,1,6.000000,0.165711,0.928889,15.666667,44.704806,1.000000,43.070000,25.440000,0.760405,1.000000,0.500000,0.166667,4.130000
1,0,mixed,1,7.000000,0.143169,0.958127,10.977143,30.797253,0.000000,62.300000,40.240000,0.776408,2.000000,0.071429,0.214286,-11.130000
9,3,reset-heavy,1,11.000000,0.091004,0.948862,5.490000,31.023207,3.000000,113.350000,39.850000,0.324220,4.000000,0.454545,0.181818,-24.610000


In [ ]:
# Try to show each representative path from a different game when possible.
representative_paths = select_representative_paths(
    clustered_possessions,
    analysis_paths,
    group_column="path_cluster",
    unique_games=UNIQUE_REPRESENTATIVE_GAMES,
)

rep_fig = plot_representative_paths(
    representative_paths,
    title=f"{TEAM_ID.title()} representative scoring path styles, {SEASON} sample",
)
rep_fig.show()

In [ ]:
top_path_possessions = clustered_possessions.copy()
top_path_source_paths = analysis_paths

if EXCLUDE_HUCKS_FROM_TOP_PATHS:
    top_path_possessions = top_path_possessions[
        top_path_possessions["huck_count"].fillna(0).eq(0)
    ].copy()
    top_path_ids = set(top_path_possessions["possession_id"])
    top_path_source_paths = [
        path for path in analysis_paths
        if path["possession_id"].iloc[0] in top_path_ids
    ]

top_paths = select_top_paths(
    top_path_possessions,
    top_path_source_paths,
    metric="aec_per_throw",
    n=3,
)

top_path_title = "highest non-huck long-field aEC per throw" if EXCLUDE_HUCKS_FROM_TOP_PATHS else "highest long-field aEC per throw"

best_fig = plot_possession_path(
    top_paths[0],
    title=f"{TEAM_ID.title()} {top_path_title} scoring possession, {SEASON} sample",
)
best_fig.show()


## Middle Non-Huck Scoring Possessions

The highest `aEC_per_throw` possession can still be an outlier. This view sorts the filtered non-huck possessions by `aEC_per_throw` and plots the middle five, which should be closer to normal efficient offense.

In [ ]:
MIDDLE_PATH_COUNT = 5
MIDDLE_PATH_METRIC = "aec_per_throw"

middle_source = top_path_possessions.sort_values(MIDDLE_PATH_METRIC).reset_index(drop=True)
middle_count = min(MIDDLE_PATH_COUNT, len(middle_source))
middle_start = max((len(middle_source) - middle_count) // 2, 0)
middle_path_possessions = middle_source.iloc[
    middle_start:middle_start + middle_count
].copy()

middle_path_lookup = {
    path["possession_id"].iloc[0]: path
    for path in top_path_source_paths
}
middle_paths = {
    f"middle {rank + 1}: {row[MIDDLE_PATH_METRIC]:.3f}": middle_path_lookup[row["possession_id"]]
    for rank, (_, row) in enumerate(middle_path_possessions.iterrows())
    if row["possession_id"] in middle_path_lookup
}

middle_path_possessions[[
    "possession_id", "GameID", "start_y", "end_y", "field_progress",
    "throw_count", "huck_count", MIDDLE_PATH_METRIC
]]


,possession_id,GameID,start_y,end_y,field_progress,throw_count,huck_count,aec_per_throw
9,2026-05-30-SD-OAK|3|8|1|True,2026-05-30-SD-OAK,11.35,102.45,91.10,10,0,0.100970
10,2026-05-30-SD-OAK|1|3|1|True,2026-05-30-SD-OAK,40.00,103.80,63.80,9,0,0.107326
11,2026-05-30-SD-OAK|3|1|1|True,2026-05-30-SD-OAK,26.19,110.00,83.81,10,0,0.109619
12,2026-04-26-ORE-OAK|4|1|1|True,2026-04-26-ORE-OAK,19.77,111.24,91.47,9,0,0.110344
13,2026-06-26-OAK-COL|3|5|1|False,2026-06-26-OAK-COL,20.00,106.45,86.45,9,0,0.111316


In [ ]:
middle_fig = plot_representative_paths(
    middle_paths,
    title=f"{TEAM_ID.title()} middle {len(middle_paths)} non-huck long-field scoring possessions, {SEASON} sample",
)
middle_fig.show()


## Compare Teams Side By Side

The readable comparison is one possession style at a time. Build representative paths for each team, then choose a style like `huck`, `reset`, `quick`, or `methodical` to compare across teams.

In [ ]:
TEAM_IDS_TO_COMPARE = ["glory", "empire", "spiders"]

team_representative_paths = {}
team_cluster_summaries = {}

for compare_team_id in TEAM_IDS_TO_COMPARE:
    compare_games = all_games[
        all_games["HomeTeamID"].str.lower().eq(compare_team_id.lower())
        | all_games["AwayTeamID"].str.lower().eq(compare_team_id.lower())
    ].reset_index(drop=True)

    if MAX_GAMES is None:
        selected_games = compare_games.copy()
    elif SAMPLE_GAMES_RANDOMLY:
        selected_games = (
            compare_games
            .sample(n=min(MAX_GAMES, len(compare_games)), random_state=RANDOM_STATE)
            .sort_values("StartTimestamp")
            .reset_index(drop=True)
        )
    else:
        selected_games = compare_games.head(MAX_GAMES).copy()

    compare_throws = fetch_shownspace_throws_for_games(
        selected_games["GameID"].tolist(),
        delay=0.15,
    )
    compare_possessions, compare_paths = build_scoring_possessions(
        compare_throws,
        team_id=compare_team_id,
    )

    compare_analysis_possessions = compare_possessions.copy()
    if PULL_RECEIVE_SCORES_ONLY:
        compare_analysis_possessions = compare_analysis_possessions[
            compare_analysis_possessions["possession_num"].eq(1)
        ].copy()
    if LONG_FIELD_ONLY:
        compare_analysis_possessions = compare_analysis_possessions[
            compare_analysis_possessions["start_y"].le(MAX_START_Y)
            & compare_analysis_possessions["field_progress"].ge(MIN_FIELD_PROGRESS)
        ].copy()

    compare_ids = set(compare_analysis_possessions["possession_id"])
    compare_analysis_paths = [
        path for path in compare_paths
        if path["possession_id"].iloc[0] in compare_ids
    ]

    compare_clustered = cluster_scoring_possessions(
        compare_analysis_possessions,
        compare_analysis_paths,
        n_clusters=4,
    )
    team_cluster_summaries[compare_team_id] = summarize_path_clusters(compare_clustered)
    team_representative_paths[compare_team_id] = select_representative_paths(
        compare_clustered,
        compare_analysis_paths,
        group_column="path_cluster",
        unique_games=UNIQUE_REPRESENTATIVE_GAMES,
    )

comparison_counts = pd.DataFrame([
    {
        "team_id": team_id,
        "representative_paths": len(representative_paths),
    }
    for team_id, representative_paths in team_representative_paths.items()
])
comparison_counts


,team_id,representative_paths
0,glory,4
1,empire,4
2,spiders,4


In [ ]:
STYLE_TO_COMPARE = "huck"  # Try "reset", "quick", "methodical", or None

style_title = (
    "All styles"
    if STYLE_TO_COMPARE is None
    else STYLE_TO_COMPARE.title()
)

comparison_fig = plot_team_representative_path_grid(
    team_representative_paths,
    title=f"{style_title} representative scoring paths by team, {SEASON}",
    style_filter=STYLE_TO_COMPARE,
    show_arrows=False,
)
comparison_fig.show()


### Optional: All Styles

This is busier, but useful as a quick overview after the one-style comparison makes sense.

In [ ]:
all_styles_fig = plot_team_representative_path_grid(
    team_representative_paths,
    title=f"All representative scoring path styles by team, {SEASON}",
    style_filter=None,
    show_arrows=False,
)
all_styles_fig.show()


## Catch Location Heatmap

This shows where completed throws in scoring possessions are caught.

In [ ]:
heatmap = plot_scoring_heatmap(
    analysis_paths,
    title=f"{TEAM_ID.title()} scoring-possession catch heatmap, {SEASON} sample",
)
heatmap.show()

## Full Team Season

After the sample plots look right, run the full team season by setting `MAX_GAMES = None` below.

In [ ]:
MAX_GAMES = None

games_full, throws_full = fetch_shownspace_season_throws(
    season=SEASON,
    team_id=TEAM_ID,
    max_games=MAX_GAMES,
    delay=0.15,
)

possessions_full, paths_full = build_scoring_possessions(throws_full, team_id=TEAM_ID)
avg_path_full = average_scoring_path(paths_full)

print(f"Games loaded: {len(games_full):,}")
print(f"Throws loaded: {len(throws_full):,}")
print(f"Scoring possessions found for {TEAM_ID}: {len(possessions_full):,}")

possessions_full.sort_values("risk_adjusted_aec_per_throw", ascending=False).head(20)

Games loaded: 10
Throws loaded: 5,671
Scoring possessions found for spiders: 257


,possession_id,GameID,team_id,start_timestamp,game_quarter,quarter_point,possession_num,is_home_team,line_type,start_x,...,mean_cp,risk_adjusted_aec_per_throw,total_yards,yards_per_throw,total_throw_distance,avg_throw_distance,max_throw_distance,huck_count,reset_count,lateral_yards
22,2026-04-26-ORE-OAK|3|8|2|True,2026-04-26-ORE-OAK,spiders,2026-04-26 15:30:00,3,8,2,True,d_line,-8.37,...,0.976773,0.976773,3.96,3.960,4.609902,4.609902,4.609902,0,0,2.36
42,2026-05-02-SLC-OAK|2|6|2|True,2026-05-02-SLC-OAK,spiders,2026-05-02 18:00:00,2,6,2,True,d_line,-18.62,...,0.973911,0.973911,3.77,3.770,5.083404,5.083404,5.083404,0,0,3.41
11,2026-04-26-ORE-OAK|2|3|2|True,2026-04-26-ORE-OAK,spiders,2026-04-26 15:30:00,2,3,2,True,d_line,6.19,...,0.972435,0.972435,6.91,6.910,6.916083,6.916083,6.916083,0,0,0.29
240,2026-06-27-OAK-SLC|2|8|2|False,2026-06-27-OAK-SLC,spiders,2026-06-27 19:00:00,2,8,2,False,d_line,6.86,...,0.964945,0.964945,8.52,8.520,9.858600,9.858600,9.858600,0,0,4.96
79,2026-05-09-OAK-SEA|4|2|2|False,2026-05-09-OAK-SEA,spiders,2026-05-09 15:00:00,4,2,2,False,d_line,13.36,...,0.953415,0.953415,7.65,7.650,12.181662,12.181662,12.181662,0,0,9.48
117,2026-05-17-SD-OAK|2|1|3|True,2026-05-17-SD-OAK,spiders,2026-05-17 15:00:00,2,1,3,True,o_line,-0.60,...,0.944834,0.944834,16.55,16.550,16.658706,16.658706,16.658706,0,0,1.90
113,2026-05-17-SD-OAK|1|4|3|True,2026-05-17-SD-OAK,spiders,2026-05-17 15:00:00,1,4,3,True,o_line,-15.95,...,0.940206,0.940206,17.82,17.820,18.661297,18.661297,18.661297,0,0,5.54
208,2026-06-26-OAK-COL|1|1|1|False,2026-06-26-OAK-COL,spiders,2026-06-26 19:00:00,1,1,1,False,o_line,-2.41,...,0.939296,0.939296,12.36,12.360,16.006202,16.006202,16.006202,0,0,10.17
74,2026-05-09-OAK-SEA|3|8|2|False,2026-05-09-OAK-SEA,spiders,2026-05-09 15:00:00,3,8,2,False,d_line,-18.96,...,0.928043,0.928043,20.53,20.530,20.539060,20.539060,20.539060,0,0,0.61
104,2026-05-10-OAK-ORE|3|7|2|False,2026-05-10-OAK-ORE,spiders,2026-05-10 14:00:00,3,7,2,False,d_line,-10.55,...,0.893104,0.893104,25.61,25.610,26.050432,26.050432,26.050432,0,0,4.77


In [ ]:
fig_full = plot_average_scoring_path(
    avg_path_full,
    paths=paths_full,
    title=f"{TEAM_ID.title()} average scoring path, {SEASON}",
    show_individual_paths=True,
)
fig_full.show()

In [ ]:
heatmap_full = plot_scoring_heatmap(
    paths_full,
    title=f"{TEAM_ID.title()} scoring-possession catch heatmap, {SEASON}",
)
heatmap_full.show()